In [1]:
from hydra import compose, initialize
from omegaconf import OmegaConf
import json
import os
from pathlib import Path
import glob
with initialize(config_path="dev/whole_body_benchmark/configs"):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

/tmp/ipykernel_65/1456265780.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="dev/whole_body_benchmark/configs"):


paths:
  nako_dir: /nfs/data/nii/data0/GNC/GNC_759
  nnUNet_dir: /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/
  results_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/results/
  data_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/data/
  oppscreen_dir: /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/data/oppscreen/



In [2]:
out_path = Path(cfg.paths.oppscreen_dir) / 'splits_966.json'
with open(out_path, 'r') as f:
    splits = json.load(f)
subjects_train, subjects_test = splits['train'], splits['test']

In [3]:
sub = splits["test"][0]
print(sub)

100161


In [4]:
# GT: /nfs/data/nii/data0/GNC/GNC_759/data/"subject"/30/opportunistic-screening/seg.nii.gz
# Pred: /nfs/data/nii/data0/GNC/GNC_759/data/"subject"/30/wholebody/subsetFW.nii.gz


In [ ]:
import sys
import numpy as np
import nibabel as nib
from scipy.ndimage import zoom
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

sys.path.insert(0, "dev/whole_body_benchmark/scripts")
from compute_metrics import dice, nsd

LABEL_MAP = {
    "AT_pelvis": 1,
    "IMAT_pelvis": 2,
    "Muscle_pelvis": 3,
    "AVAT_pelvis": 4,
    "AT_upper_abdomen": 5,
    "IMAT_upper_abdomen": 6,
    "Muscle_upper_abdomen": 7,
    "AVAT_upper_abdomen": 8,
    "AT_thorax": 9,
    "IMAT_thorax": 10,
    "Muscle_thorax": 11,
    "AVAT_thorax": 12,
    "heart": 13,
    "IVD": 14,
    "vertebra_body": 15,
    "vertebra_posterior_elements": 16,
    "liver": 17,
    "pankreas": 18,
    "aorta": 19,
    "kidney_cortex_left": 20,
    "kidney_hilus_left": 21,
    "kidney_medulla_left": 22,
    "kidney_cortex_right": 23,
    "kidney_hilus_right": 24,
    "kidney_medulla_right": 25,
}

def eval_subject(subject, nako_dir, tolerance_mm=2.0):
    gt_path   = f"{nako_dir}/data/{subject}/30/opportunistic-screening/seg.nii.gz"
    pred_path = f"{nako_dir}/data/{subject}/30/wholebody/subsetFW.nii.gz"
    try:
        gt_nii   = nib.load(gt_path)
        pred_nii = nib.load(pred_path)
    except Exception as e:
        return subject, {"error": str(e)}

    gt   = np.asarray(gt_nii.dataobj, dtype=np.int32)
    pred = np.asarray(pred_nii.dataobj, dtype=np.int32)
    if pred.ndim == 4:
        pred = pred[..., 0]

    if pred.shape != gt.shape:
        zoom_factors = tuple(g / p for g, p in zip(gt.shape, pred.shape))
        pred = zoom(pred, zoom_factors, order=0)

    voxel_spacing = gt_nii.header.get_zooms()[:3]
    results = {}
    for label_name, k in LABEL_MAP.items():
        gt_k   = gt   == k
        pred_k = pred == k
        d = dice(gt_k, pred_k)
        n = nsd(gt_k, pred_k, voxel_spacing, tolerance_mm)
        results[label_name] = {"dice": float(d), "nsd": float(n)}
    return subject, results


nako_dir     = cfg.paths.nako_dir
n_workers    = 16
tolerance_mm = 2.0

all_results = {}
with ProcessPoolExecutor(max_workers=n_workers) as executor:
    futures = {
        executor.submit(eval_subject, sub, nako_dir, tolerance_mm): sub
        for sub in subjects_test
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Evaluating test set"):
        subject, res = future.result()
        all_results[subject] = res

print(f"Evaluated {len(all_results)} / {len(subjects_test)} subjects")


Evaluating test set:   0%|          | 0/290 [00:00<?, ?it/s]

: 

: 

KeyboardInterrupt: 

In [ ]:
python scripts/eval.py --splits /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/data/oppscreen/splits_966.json  --nako-dir /nfs/data/nii/data0/GNC/GNC_759 --no-nsd --workers 16

{'100161': {'error': "No such file or no access: '/nfs/data/nii/data0/GNC/GNC_759/data/100161/30/opportunistic-screening/seg.nii.gz'"},
 '100963': {'error': "No such file or no access: '/nfs/data/nii/data0/GNC/GNC_759/data/100963/30/opportunistic-screening/seg.nii.gz'"},
 '100263': {'error': "No such file or no access: '/nfs/data/nii/data0/GNC/GNC_759/data/100263/30/opportunistic-screening/seg.nii.gz'"},
 '100847': {'error': "No such file or no access: '/nfs/data/nii/data0/GNC/GNC_759/data/100847/30/opportunistic-screening/seg.nii.gz'"},
 '100784': {'error': "No such file or no access: '/nfs/data/nii/data0/GNC/GNC_759/data/100784/30/opportunistic-screening/seg.nii.gz'"},
 '100193': {'error': "No such file or no access: '/nfs/data/nii/data0/GNC/GNC_759/data/100193/30/opportunistic-screening/seg.nii.gz'"},
 '100500': {'error': "No such file or no access: '/nfs/data/nii/data0/GNC/GNC_759/data/100500/30/opportunistic-screening/seg.nii.gz'"},
 '100256': {'error': "No such file or no access: